                                   # Item Catalog Creation for Recommendation System
 ## Objective
To standardize item-level data by resolving inconsistencies in item names and categories, and to generate a canonical item catalog with aggregated statistics.
## Data Loading
The raw dataset (main_data.csv) is loaded from the specified directory.
Column names are cleaned to remove unwanted spaces.
Initial inspection is performed to understand data shape and structure.
#Data Cleaning and Normalization
itemId is converted to numeric format.
itemName is cleaned using text normalization (removing extra spaces).
category is standardized (capitalization and formatting).
Rows with missing values in key fields are removed.
## Data Validation
Ensures required columns exist: itemId, itemName, category.
Invalid or missing entries are filtered out.
Data types are enforced (e.g., itemId as integer).
## Aggregation Logic
For each unique itemId:
The most frequent itemName is selected as the canonical name.
The most frequent category is selected as the canonical category.
Counts of name and category variations are tracked.
Total occurrences of the item are calculated.
## Variant Detection
nameVariantCount: number of unique names per item
categoryVariantCount: number of unique categories per item
reviewFlag:
1 → if inconsistencies exist
0 → if data is consistent
## Item Catalog Creation
A final catalog is generated with the following fields:
itemId
itemName (canonical)
category (canonical)
avgPrice, minPrice, maxPrice
totalRowsSeen
nameVariantCount
categoryVariantCount
reviewFlag
## Benefits
Ensures data consistency across the system
Removes ambiguity in item identity
Improves recommendation accuracy
Enables reliable feature engineering
Supports downstream embedding and modeling

## Cell-wise Explanation (Easy)
- Cell 1: Project objective and expected final outputs.
- Cell 2: Import section heading.
- Cell 3: Required libraries import (`pandas`, `numpy`, `re`, `Path`, `Counter`).
- Cell 4: Data path setup heading.
- Cell 5: Raw file path check (`exists`, path print).
- Cell 6: Dataset load heading.
- Cell 7: Main dataset load, columns clean, shape/preview display.
- Cell 8: Text normalization heading.
- Cell 9: `normalize_text()` and `normalize_category()` helper functions.
- Cell 10: Cleaning and validation heading.
- Cell 11: Required column validation, type cleanup, basic cleaning.
- Cell 12: Exploration heading.
- Cell 13: Unique counts (itemId, itemName, category).
- Cell 14: Frequency counter heading.
- Cell 15: Item-wise name/category/price counters build.
- Cell 16: Item catalog build heading.
- Cell 17: Canonical catalog dataframe build.
- Cell 18: Review-flag items filter and display.
- Cell 19: Conflict analysis heading.
- Cell 20: Conflict details table generation.
- Cell 21: Save output heading.
- Cell 22: Catalog, review, conflict CSV save.
- Cell 23: Final verification heading.
- Cell 24: Saved catalog reload and final preview.
- Cell 25: Empty/placeholder cell.

### Possible Outputs
- Shape, columns, unique counts, conflict/review counts.
- `item_catalog.csv`, `item_catalog_needs_review.csv`, `item_catalog_conflict_details.csv` saved messages.

# Import Libraries

### Cell 3: Import Libraries
**Explanation:** Data cleaning এবং catalog তৈরির জন্য প্রয়োজনীয় Python libraries import করে।

**Possible Output:** সাধারণত কোনো output নেই; imports success হলে proceed করবে।

In [1]:
import re
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

# Data Path Setup

### Cell 5: Data Path Setup
**Explanation:** Raw dataset path configure করে file exists check করে।

**Possible Output:** `File exists: True/False` এবং file path print।

In [4]:
DATA_DIR = Path(r"D:\recommendation_item_API\data")

RAW_FILE = DATA_DIR / "main_data.csv"

print("File exists:", RAW_FILE.exists())
print("Path:", RAW_FILE)

File exists: True
Path: D:\recommendation_item_API\data\main_data.csv


# Load Dataset

### Cell 7: Load Dataset
**Explanation:** `main_data.csv` load করে columns clean করে sample data দেখায়।

**Possible Output:** Shape, column names, এবং head preview।

In [16]:
df = pd.read_csv(RAW_FILE)

df.columns = [c.strip() for c in df.columns]

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())

Shape: (40886, 32)
Columns:
['Transaction_ID', 'Customer_ID', 'Timestamp', 'Product_Category', 'Product_Name', 'Quantity', 'Price', 'Total_Amount', 'Day_Type', 'transactionId', 'customerId', 'customerName', 'customerPersona', 'customerSegment', 'itemId', 'itemName', 'category', 'quantity', 'price', 'totalAmount', 'orderDate', 'isHoliday', 'isFestival', 'isRamadan', 'isEidPrep', 'isEid', 'season', 'timeSlot', 'dayType', 'dayOfWeek', 'hour', 'month']


,Transaction_ID,Customer_ID,Timestamp,Product_Category,Product_Name,Quantity,Price,Total_Amount,Day_Type,transactionId,...,isFestival,isRamadan,isEidPrep,isEid,season,timeSlot,dayType,dayOfWeek,hour,month
0,1,23561,2025-01-01 07:29:00,Dairy-Other,Herman Peanut Butter-340gm,1,154.46,154.46,Weekday,1,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,7,1
1,1,23561,2025-01-01 07:29:00,Bakery-Bread,Pran Toast-250g,4,63.66,254.64,Weekday,1,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,7,1
2,1,23561,2025-01-01 07:29:00,Beverage-Hot,Horlicks Classic Malt-1Kg Jar,1,329.90,329.90,Weekday,1,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,7,1
3,2,23569,2025-01-01 11:33:00,Pantry-DryFood,Lassa Special Semai-500g(Pkt),2,155.51,311.02,Weekday,2,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,11,1
4,2,23569,2025-01-01 11:33:00,Fruits-Fresh,Atta Fruits,2,46.66,93.32,Weekday,2,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,11,1


# Text Normalization 

### Cell 9: Text Normalization Helpers
**Explanation:** `normalize_text()` এবং `normalize_category()` define করে naming consistency আনে।

**Possible Output:** Direct output নেই; helper functions প্রস্তুত হয়।

In [17]:
def normalize_text(x):
    if pd.isna(x):
        return x
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x

def normalize_category(cat):
    if pd.isna(cat):
        return cat
    cat = normalize_text(cat)
    parts = [p.strip() for p in cat.split("-")]
    parts = [p.capitalize() if p else p for p in parts]
    return "-".join(parts)

# Data Cleaning and Validation

### Cell 11: Data Cleaning and Validation
**Explanation:** Required columns আছে কিনা verify করে এবং null/type issues clean করে।

**Possible Output:** Missing column থাকলে error; না হলে cleaned dataframe state।

In [18]:
required_cols = ["itemId", "itemName", "category"]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"{col} column not found in main_data.csv")

df["itemId"] = pd.to_numeric(df["itemId"], errors="coerce")
df["itemName"] = df["itemName"].apply(normalize_text)
df["category"] = df["category"].apply(normalize_category)

df = df.dropna(subset=["itemId", "itemName", "category"]).copy()
df["itemId"] = df["itemId"].astype(int)

print("Shape after basic cleaning:", df.shape)
display(df[["itemId", "itemName", "category"]].head())

Shape after basic cleaning: (40886, 32)


,itemId,itemName,category
0,7075,Herman Peanut Butter-340gm,Dairy-Other
1,27,Pran Toast-250g,Bakery-Bread
2,6814,Horlicks Classic Malt-1Kg Jar,Beverage-Hot
3,25474,Lassa Special Semai-500g(Pkt),Pantry-Dryfood
4,15131,Atta Fruits,Fruits-Fresh


# Basic Data Exploration


### Cell 13: Basic Data Exploration
**Explanation:** Unique item, item name, category count চেক করে data diversity বোঝে।

**Possible Output:** Unique count print lines.

In [19]:
print("Unique itemId count:", df["itemId"].nunique())
print("Unique itemName count:", df["itemName"].nunique())
print("Unique category count:", df["category"].nunique())

Unique itemId count: 229
Unique itemName count: 229
Unique category count: 43


# Build Item Frequency Counters

### Cell 15: Build Frequency Counters
**Explanation:** Item-wise নাম, category, price frequency accumulate করে counters তৈরি করে।

**Possible Output:** সাধারণত direct output নেই; counters populate হয়।

In [20]:
item_name_counter = defaultdict(Counter)
item_category_counter = defaultdict(Counter)
item_price_values = defaultdict(list)

for _, row in df.iterrows():
    iid = int(row["itemId"])
    item_name_counter[iid][row["itemName"]] += 1
    item_category_counter[iid][row["category"]] += 1
    
    if "price" in df.columns and pd.notna(row.get("price", np.nan)):
        try:
            item_price_values[iid].append(float(row["price"]))
        except:
            pass

#  Create Item Catalog

### Cell 17: Create Item Catalog
**Explanation:** প্রতিটি `itemId`-এর canonical name/category বেছে final catalog dataframe build করে।

**Possible Output:** `item_catalog_df` তৈরি হয়; preview/display থাকতে পারে।

In [21]:
catalog_rows = []

for iid in sorted(item_name_counter.keys()):
    canonical_name = item_name_counter[iid].most_common(1)[0][0]
    canonical_category = item_category_counter[iid].most_common(1)[0][0]
    
    name_variants = len(item_name_counter[iid])
    category_variants = len(item_category_counter[iid])
    
    total_rows = sum(item_name_counter[iid].values())
    
    if len(item_price_values[iid]) > 0:
        avg_price = round(float(np.mean(item_price_values[iid])), 2)
        min_price = round(float(np.min(item_price_values[iid])), 2)
        max_price = round(float(np.max(item_price_values[iid])), 2)
    else:
        avg_price = np.nan
        min_price = np.nan
        max_price = np.nan
    
    catalog_rows.append({
        "itemId": iid,
        "itemName": canonical_name,
        "category": canonical_category,
        "avgPrice": avg_price,
        "minPrice": min_price,
        "maxPrice": max_price,
        "totalRowsSeen": total_rows,
        "nameVariantCount": name_variants,
        "categoryVariantCount": category_variants,
        "reviewFlag": 1 if (name_variants > 1 or category_variants > 1) else 0
    })

item_catalog_df = pd.DataFrame(catalog_rows)

print("item_catalog_df shape:", item_catalog_df.shape)
display(item_catalog_df.head(20))

item_catalog_df shape: (229, 10)


,itemId,itemName,category,avgPrice,minPrice,maxPrice,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag
0,13,Lemon Bright Dish Wash 250ml B2 G1,Household-Kitchen,121.34,121.34,121.34,115,1,1,0
1,27,Pran Toast-250g,Bakery-Bread,63.66,63.66,63.66,244,1,1,0
2,109,Wheel 2in1 Clean & Fresh Powder-1Kg,Personal-Care-Cosmetics,587.26,587.26,587.26,65,1,1,0
3,125,Wheel Laundry Soap 125g,Personal-Care-Bath,122.60,122.60,122.60,43,1,1,0
4,129,Vim Bar Lemon-125gm,Household-Kitchen,73.72,73.72,73.72,144,1,1,0
5,133,Rin Advanced-500gm,Household-Laundry,114.83,114.83,114.83,375,1,1,0
6,292,Super White Powder-1000g,Personal-Care-Cosmetics,187.42,187.42,187.42,62,1,1,0
7,296,Chaka Advanced Ball Soap-125gm,Personal-Care-Bath,74.71,74.71,74.71,29,1,1,0
8,318,Musur Pulse Kangaroo -1Kg (sp),Pantry-Pulses,271.84,271.84,271.84,127,1,1,0
9,332,ACI Pure Salt-1Kg,Spices-Cooking,239.78,238.37,250.29,278,1,1,0


### Cell 18: Review Flag Items
**Explanation:** `reviewFlag == 1` থাকা items আলাদা করে review list বানায়।

**Possible Output:** `Items needing review: N` + sample rows।

In [22]:
needs_review_df = item_catalog_df[item_catalog_df["reviewFlag"] == 1].copy()

print("Items needing review:", needs_review_df.shape[0])
display(needs_review_df.head(20))

Items needing review: 0


,itemId,itemName,category,avgPrice,minPrice,maxPrice,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag


#  Conflict Details Analysis

### Cell 20: Conflict Details Analysis
**Explanation:** যেসব itemId তে name/category conflict আছে সেগুলোর detailed table তৈরি করে।

**Possible Output:** Conflict detail dataframe preview।

In [26]:
conflict_rows = []

for iid in sorted(item_name_counter.keys()):
    if len(item_name_counter[iid]) > 1 or len(item_category_counter[iid]) > 1:
        conflict_rows.append({
            "itemId": iid,
            "itemNameVariants": dict(item_name_counter[iid]),
            "categoryVariants": dict(item_category_counter[iid])
        })

conflict_details_df = pd.DataFrame(conflict_rows)

print("Conflict details shape:", conflict_details_df.shape)
display(conflict_details_df.head(20))

Conflict details shape: (0, 0)


""


#  Save Output Files

### Cell 22: Save Output Files
**Explanation:** Item catalog, review list, conflict details CSV ফাইলে save করে।

**Possible Output:** Saved file path/messages print হতে পারে।

In [27]:
ITEM_CATALOG_FILE = DATA_DIR / "item_catalog.csv"
REVIEW_FILE = DATA_DIR / "item_catalog_needs_review.csv"
CONFLICT_FILE = DATA_DIR / "item_catalog_conflict_details.csv"

item_catalog_df.to_csv(ITEM_CATALOG_FILE, index=False)
needs_review_df.to_csv(REVIEW_FILE, index=False)
conflict_details_df.to_csv(CONFLICT_FILE, index=False)

print("Saved:", ITEM_CATALOG_FILE)
print("Saved:", REVIEW_FILE)
print("Saved:", CONFLICT_FILE)

Saved: D:\recommendation_item_API\data\item_catalog.csv
Saved: D:\recommendation_item_API\data\item_catalog_needs_review.csv
Saved: D:\recommendation_item_API\data\item_catalog_conflict_details.csv


# Final Catalog Verification

### Cell 24: Final Catalog Verification
**Explanation:** Saved `item_catalog.csv` আবার load করে final shape এবং sample rows verify করে।

**Possible Output:** `Final catalog shape: ...` + dataframe preview.

In [28]:
final_catalog = pd.read_csv(ITEM_CATALOG_FILE)

print("Final catalog shape:", final_catalog.shape)
display(final_catalog.head(20))

Final catalog shape: (229, 10)


,itemId,itemName,category,avgPrice,minPrice,maxPrice,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag
0,13,Lemon Bright Dish Wash 250ml B2 G1,Household-Kitchen,121.34,121.34,121.34,115,1,1,0
1,27,Pran Toast-250g,Bakery-Bread,63.66,63.66,63.66,244,1,1,0
2,109,Wheel 2in1 Clean & Fresh Powder-1Kg,Personal-Care-Cosmetics,587.26,587.26,587.26,65,1,1,0
3,125,Wheel Laundry Soap 125g,Personal-Care-Bath,122.60,122.60,122.60,43,1,1,0
4,129,Vim Bar Lemon-125gm,Household-Kitchen,73.72,73.72,73.72,144,1,1,0
5,133,Rin Advanced-500gm,Household-Laundry,114.83,114.83,114.83,375,1,1,0
6,292,Super White Powder-1000g,Personal-Care-Cosmetics,187.42,187.42,187.42,62,1,1,0
7,296,Chaka Advanced Ball Soap-125gm,Personal-Care-Bath,74.71,74.71,74.71,29,1,1,0
8,318,Musur Pulse Kangaroo -1Kg (sp),Pantry-Pulses,271.84,271.84,271.84,127,1,1,0
9,332,ACI Pure Salt-1Kg,Spices-Cooking,239.78,238.37,250.29,278,1,1,0
